In [1]:
#downloading the historic data
import os
import yfinance as yf
import pandas as pd
import requests
from datetime import datetime, timedelta
from io import StringIO
import logging
logging.getLogger("yfinance").setLevel(logging.CRITICAL)

# CONFIG
# ========================
OUTPUT_FILE = "sp500_historical_data.csv"
TICKERS_FILE = "sp500_tickers.csv"
API_KEY = "68b88d76172529.85745688"  # Your EODHD API token

# FUNCTIONS
# ========================
def get_sp500_tickers_from_wikipedia():
    """Fetch S&P 500 tickers from Wikipedia"""
    try:
        url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
        #tables = pd.read_html(requests.get(url, headers={'User-agent': 'Mozilla/5.0'}).text)
        html = requests.get(url, headers={'User-agent': 'Mozilla/5.0'}).text
        tables = pd.read_html(StringIO(html))
        sp500_table = tables[0]
        tickers = sp500_table['Symbol'].str.replace(".", "-", regex=False).tolist()
        print(f" Loaded {len(tickers)} tickers from Wikipedia.")
        return tickers
    except Exception as e:
        print(f" Failed to load tickers from Wikipedia: {e}")
        return []

def get_sp500_tickers_from_eodhd(api_key):
    """Fetch S&P 500 tickers from EODHD API"""
    try:
        url = f"https://eodhd.com/api/eod/SPY.US?api_token={api_key}&fmt=json"
        resp = requests.get(url)
        data = resp.json()
        if isinstance(data, list):
            tickers = [item['code'].replace(".", "-") for item in data]
            print(f" Loaded {len(tickers)} tickers from EODHD API.")
            return tickers
        else:
            print(f" Unexpected response from EODHD API: {data}")
            return []
    except Exception as e:
        print(f" Failed to load tickers from EODHD API: {e}")
        return []

def load_tickers():
    """Load tickers from CSV, else fallback to Wikipedia, else API. 
       Save to CSV if fetched from web."""
    tickers = []

    # 1. Try CSV
    if os.path.exists(TICKERS_FILE):
        try:
            tickers_df = pd.read_csv(TICKERS_FILE)
            if "Ticker" in tickers_df.columns:
                tickers = tickers_df["Ticker"].str.replace(".", "-", regex=False).tolist()
                print(f" Loaded {len(tickers)} tickers from {TICKERS_FILE}.")
        except Exception as e:
            print(f" Could not read tickers from {TICKERS_FILE}: {e}")

    # 2. Fallback to Wikipedia
    if not tickers:
        tickers = get_sp500_tickers_from_wikipedia()
        if tickers:
            pd.DataFrame({"Ticker": tickers}).to_csv(TICKERS_FILE, index=False)
            print(f" Saved {len(tickers)} tickers to {TICKERS_FILE} (from Wikipedia).")

    # 3. Fallback to EODHD API
    if not tickers:
        tickers = get_sp500_tickers_from_eodhd(API_KEY)
        if tickers:
            pd.DataFrame({"Ticker": tickers}).to_csv(TICKERS_FILE, index=False)
            print(f" Saved {len(tickers)} tickers to {TICKERS_FILE} (from EODHD API).")

    # 4. Exit if none found
    if not tickers:
        print(" No tickers available from any source. Exiting.")
        exit(1)

    return tickers
# MAIN SCRIPT
# ========================

# Load tickers (CSV → Wikipedia → EODHD API)
tickers = load_tickers()

# Check for existing data and determine the start date for new downloads
existing_data = None
start_date = "2020-01-01"
if os.path.exists(OUTPUT_FILE):
    try:
        existing_data = pd.read_csv(OUTPUT_FILE, parse_dates=["Date"])
        if not existing_data.empty:
            last_date = pd.to_datetime(existing_data["Date"].max()).normalize()
            today = pd.to_datetime(datetime.today().date())
            if last_date >= today - timedelta(days=1):
                print(f" Existing data found. Last date: {last_date.date()}. File is already up to date.")
                exit(0)  # stop script, nothing new to fetch
            else:
                start_date = (last_date + timedelta(days=1)).strftime("%Y-%m-%d")
                print(f" Existing data found. Last date: {last_date.date()}. Downloading from {start_date}.")
        else:
            print(" Existing data file is empty. Starting fresh download.")
    except Exception as e:
        print(f" Error reading existing data: {e}. Starting fresh download.")
        existing_data = None

end_date = datetime.today().strftime("%Y-%m-%d")
new_data = []
failed_tickers = []

for ticker in tickers:
    try:
        stock = yf.Ticker(ticker)
        df = stock.history(start=start_date, end=end_date, interval="1d", auto_adjust=False)
        if not df.empty:
            df = df.reset_index()
            df["Ticker"] = ticker
            new_data.append(df)
            #print(f" Downloaded new data for {ticker} from {start_date} to {end_date}")
        else:
            #print(f" No new data for {ticker} from {start_date} to {end_date}")
            failed_tickers.append(ticker)
    except Exception as e:
        print(f" Error downloading {ticker}: {e}")
        failed_tickers.append(ticker)

# Log failed tickers
if failed_tickers:
    print(f"\n Failed tickers: {failed_tickers}")

# Concatenate and process data
if new_data:
    new_data_df = pd.concat(new_data, ignore_index=True)
    if existing_data is not None and not existing_data.empty:
        combined_data = pd.concat([existing_data, new_data_df], ignore_index=True)
        combined_data.drop_duplicates(subset=['Date', 'Ticker'], keep='last', inplace=True)
        all_data = combined_data
    else:
        all_data = new_data_df

    # Ensure 'Date' is in datetime format and sort
    all_data['Date'] = pd.to_datetime(all_data['Date'], utc=True).dt.tz_localize(None)
    all_data['Date'] = all_data['Date'].dt.strftime('%Y-%m-%d %H:%M:%S')
    all_data = all_data.sort_values(['Ticker', 'Date'])

    # Save with consistent columns
    columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits', 'Ticker']
    all_data = all_data[columns]
    all_data.to_csv(OUTPUT_FILE, index=False)

    print(f"\n Saved/Updated {OUTPUT_FILE}")
    print(f"Total rows in file: {len(all_data)}")
    print(f"Unique tickers: {all_data['Ticker'].nunique()}")
    print(f"Last date in file: {all_data['Date'].max()}")
else:
    if existing_data is not None:
        print("No new data to download. File is up to date.")
    else:
        print("Error: No data downloaded at all.")
        exit(1)


 Loaded 503 tickers from sp500_tickers.csv.
 Existing data found. Last date: 2025-09-05. File is already up to date.

 Saved/Updated sp500_historical_data.csv
Total rows in file: 1421753
Unique tickers: 506
Last date in file: 2025-09-05 04:00:00


In [2]:
#downloading tickers
import pandas as pd
import requests
from io import StringIO

# Scrape S&P 500 tickers from Wikipedia with headers
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "Mozilla/5.0"}
html = requests.get(url, headers=headers).text

# Wrap html in StringIO to avoid FutureWarning
tables = pd.read_html(StringIO(html))
sp500_table = tables[0]

# Extract ticker symbols
tickers = sp500_table["Symbol"].tolist()

# Save to CSV
output_file = "sp500_tickers.csv"
pd.DataFrame(tickers, columns=["Ticker"]).to_csv(output_file, index=False)

print(f"Collected {len(tickers)} tickers.")
print(f"Saved to {output_file}.")

Collected 503 tickers.
Saved to sp500_tickers.csv.


In [ ]:
#cleaning the historic data
import pandas as pd
import numpy as np
import boto3
import os
import yfinance as yf

USE_UNADJUSTED_CLOSE = True

SP500_SECTORS = {
    'AAPL': 'Technology',
    'MSFT': 'Technology',
    'GOOGL': 'Communication Services',
    'GOOG': 'Communication Services',
    'AMZN': 'Consumer Discretionary',
    'META': 'Communication Services',
    'TSLA': 'Consumer Discretionary',
    'NVDA': 'Information Technology',
    'JPM': 'Financials',
    'JNJ': 'Health Care',
    'V': 'Information Technology',
    'PG': 'Consumer Staples',
    'HD': 'Consumer Discretionary',
    'XOM': 'Energy',
    'UNH': 'Health Care',
    'MA': 'Information Technology',
    'BAC': 'Financials',
    'WMT': 'Consumer Staples',
    'LLY': 'Health Care',
    'DIS': 'Communication Services',
    'CMCSA': 'Communication Services',
    'KO': 'Consumer Staples',
    'PFE': 'Health Care',
    'TMO': 'Health Care',
    'MRK': 'Health Care',
    'NKE': 'Consumer Discretionary',
    'COST': 'Consumer Staples',
    'CRM': 'Information Technology',
    'AVGO': 'Information Technology',
    'PEP': 'Consumer Staples',
    'VZ': 'Communication Services',
    'ADBE': 'Information Technology',
    'MCD': 'Consumer Discretionary',
    'ABT': 'Health Care',
    'WFC': 'Financials',
    'ORCL': 'Information Technology',
    'DHR': 'Health Care',
    'T': 'Communication Services',
    'CSCO': 'Information Technology',
    'NEE': 'Utilities',
    'INTC': 'Information Technology',
    'NFLX': 'Communication Services',
    'ACN': 'Information Technology',
    'PM': 'Consumer Staples',
    'HON': 'Industrials',
    'QCOM': 'Information Technology',
    'SBUX': 'Consumer Discretionary',
    'AMGN': 'Health Care',
    'TXN': 'Information Technology',
    'RTX': 'Industrials',
    'GE': 'Industrials',
    'IBM': 'Information Technology',
    'MDT': 'Health Care',
    'CVS': 'Health Care',
    'MS': 'Financials',
    'LOW': 'Consumer Discretionary',
    'UNP': 'Industrials',
    'GS': 'Financials',
    'NOW': 'Information Technology',
    'CAT': 'Industrials',
    'GILD': 'Health Care',
    'AXP': 'Financials',
    'PYPL': 'Information Technology',
    'SCHW': 'Financials',
    'C': 'Financials',
    'UBER': 'Industrials',
    'BA': 'Industrials',
    'AMD': 'Information Technology',
    'INTU': 'Information Technology',
    'CHTR': 'Communication Services',
    'BLK': 'Financials',
    'SPG': 'Real Estate',
    'ADSK': 'Information Technology',
    'EL': 'Consumer Staples',
    'MDLZ': 'Consumer Staples',
    'ISRG': 'Health Care',
    'SYK': 'Health Care',
    'SO': 'Utilities',
    'TJX': 'Consumer Discretionary',
    'CL': 'Consumer Staples',
    'PNC': 'Financials',
    'AMT': 'Real Estate',
    'LMT': 'Industrials',
    'CME': 'Financials',
    'BX': 'Financials',
    'COP': 'Energy',
    'MRNA': 'Health Care',
    'ADI': 'Information Technology',
    'MU': 'Information Technology',
    'DLR': 'Real Estate',
    'AON': 'Financials',
    'EW': 'Health Care',
    'ETN': 'Industrials',
    'FISV': 'Information Technology',
    'CB': 'Financials',
    'EQIX': 'Real Estate',
    'EOG': 'Energy',
    'ROP': 'Industrials',
    'SNPS': 'Information Technology',
    'CI': 'Health Care',
    'APD': 'Materials',
    'CCI': 'Real Estate',
    'BIIB': 'Health Care',
    'CDNS': 'Information Technology',
    'ODFL': 'Industrials',
    'D': 'Utilities',
    'MCO': 'Financials',
    'REGN': 'Health Care',
    'TDY': 'Industrials',
    'VRTX': 'Health Care',
    'SRE': 'Utilities',
    'DXC': 'Information Technology',
    'DLTR': 'Consumer Discretionary',
    'ALLE': 'Industrials',
    'WST': 'Health Care',
    'ANSS': 'Information Technology',
    'CTAS': 'Industrials',
    'HSIC': 'Health Care',
    'TSCO': 'Consumer Discretionary',
    'LUV': 'Industrials',
    'PAYC': 'Information Technology',
    'EXR': 'Real Estate',
    'PNR': 'Industrials',
    'O': 'Real Estate',
    'HCA': 'Health Care',
    'FMC': 'Materials',
    'AFL': 'Financials',
    'BWA': 'Consumer Discretionary',
    'STX': 'Information Technology',
    'CBOE': 'Financials',
    'WRB': 'Financials',
    'KMX': 'Consumer Discretionary',
    'UHS': 'Health Care',
    'CPRT': 'Industrials',
    'JBHT': 'Industrials',
    'JCI': 'Industrials',
    'AIV': 'Real Estate',
    'FDS': 'Information Technology',
    'HWM': 'Industrials',
    'CZR': 'Consumer Discretionary',
    'VFC': 'Consumer Discretionary',
    'ROL': 'Industrials',
    'WEC': 'Utilities',
    'CINF': 'Financials',
    'DISH': 'Communication Services',
    'WHR': 'Consumer Discretionary',
    'TROW': 'Financials',
    'LVS': 'Consumer Discretionary',
    'NVR': 'Consumer Discretionary',
    'APA': 'Energy',
    'ALK': 'Industrials',
    'VNT': 'Industrials',
    'KIM': 'Real Estate',
    'EIX': 'Utilities',
    'AOS': 'Industrials',
    'EXPD': 'Industrials',
    'RJF': 'Financials',
    'URI': 'Industrials',
    'GPC': 'Consumer Discretionary',
    'NLOK': 'Information Technology',
    'SJM': 'Consumer Staples',
    'HPE': 'Information Technology',
    'MGM': 'Consumer Discretionary',
    'MKTX': 'Financials',
    'AKAM': 'Information Technology',
    'SNA': 'Industrials',
    'L': 'Financials',
    'HII': 'Industrials',
    'WAB': 'Industrials',
    'TFC': 'Financials',
    'PNW': 'Utilities',
    'LNC': 'Financials',
    'HRL': 'Consumer Staples',
    'K': 'Consumer Staples',
    'EFX': 'Industrials',
    'CMA': 'Financials',
    'WDC': 'Information Technology',
    'AAL': 'Industrials',
    'KSS': 'Consumer Discretionary',
    'F': 'Consumer Discretionary',
    'JNPR': 'Information Technology',
    'FLS': 'Industrials'
}
def get_company_sector(ticker):
    if ticker in SP500_SECTORS:
        return SP500_SECTORS[ticker]
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        sector = info.get('sector', 'Unknown')
        if not sector or sector.lower() == 'unknown':
            print(f"[!] yfinance did not return a valid sector for {ticker}.")
            return "Unknown"
        #print(f"[*] Found sector for {ticker} using yfinance: {sector}")
        return sector
    except Exception as e:
        print(f"[!] Could not find sector for {ticker} using yfinance: {e}")
        return "Unknown"

# Step 1: Load and validate historical data
try:
    df = pd.read_csv("sp500_historical_data.csv", low_memory=False, parse_dates=["Date"])
    print(f"Loaded {len(df)} rows from sp500_historical_data.csv.")
except FileNotFoundError:
    print("Error: sp500_historical_data.csv not found.")
    exit(1)

expected_cols = ["Date", "Ticker", "Open", "High", "Low", "Close", "Volume"]
if not all(col in df.columns for col in expected_cols):
    print(f"Error: Missing expected columns. Found: {df.columns.tolist()}")
    exit(1)

# Step 2: Handle 'Date' column
#print("Unique dates before conversion:", df['Date'].unique()[:10])
df['Date'] = pd.to_datetime(df['Date'], errors='coerce').dt.strftime('%Y-%m-%d')
#print("Unique dates after conversion:", df['Date'].dropna().unique()[:10])
df = df.dropna(subset=['Date'])

# Step 3: Close price handling
if USE_UNADJUSTED_CLOSE and "Adj Close" in df.columns:
    df["Close"] = df["Close"]
elif not USE_UNADJUSTED_CLOSE and "Adj Close" in df.columns:
    df["Close"] = df["Adj Close"]

# Step 4: Handle missing values
numeric_cols = ["Open", "High", "Low", "Close", "Volume"]
df[numeric_cols] = df[numeric_cols].ffill()
df = df.dropna(subset=["Ticker", "Close"])

# Step 5: Remove duplicates
initial_rows = len(df)
df = df.drop_duplicates(subset=["Date", "Ticker"])
duplicates_dropped = initial_rows - len(df)
print(f"Dropped {duplicates_dropped} duplicate rows.")

# Step 6: Remove extreme outliers (disabled for now)
if len(df) > 50:
    df["Price_Median"] = df.groupby("Ticker")["Close"].transform("median")
    # df = df[df["Close"] <= 10 * df["Price_Median"]]  # Commented out
    df = df.drop(columns=["Price_Median"])

# Step 7: Add sector info
try:
    unique_tickers = df["Ticker"].unique()
    sector_map = {ticker: get_company_sector(ticker) for ticker in unique_tickers}
    df["Sector"] = df["Ticker"].map(sector_map)
    df["Sector"] = df["Sector"].fillna("Unknown")
except Exception as e:
    print(f"Error getting sectors: {e}")
    df["Sector"] = "Unknown"

# Step 8: Add MA50
df["MA50"] = df.groupby("Ticker")["Close"].transform(lambda x: x.rolling(window=50, min_periods=1).mean())

# Step 9: Add Prev_Close & Pct_Change
df = df.sort_values(["Ticker", "Date"])
df["Prev_Close"] = df.groupby("Ticker")["Close"].shift(1)
df["Pct_Change"] = ((df["Close"] - df["Prev_Close"]) / df["Prev_Close"]) * 100

# Step 10: Select final columns
df = df[[
    "Date", "Ticker", "Open", "High", "Low", "Close", "Volume",
    "Sector", "MA50", "Prev_Close", "Pct_Change"
]]

# Step 11: Save locally
df.to_csv("sp500_cleaned_data.csv", index=False)
df.to_parquet("sp500_cleaned_data.parquet")

print(f"Saved cleaned file with {len(df)} rows and {df['Ticker'].nunique()} tickers.")
print(f"Last date in cleaned data: {df['Date'].max()}")

In [ ]:
#forecasting future trends without sentiments
import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings("ignore")

# ===== CONFIG =====
FORECAST_HORIZON_DAYS = 10
MIN_HISTORY_DAYS = 50
SELECTED_TICKERS = [ "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA",
    "META", "TSLA", "PLTR", "NFLX", "AMD", "AVGO", "LLY", "GME", "UAL", "CCL", "DASH"]
RECENT_DAYS = 180

def prepare_data(df, ticker):
    """
    Prepare historical data for Prophet with extra regressors (no sentiment).
    """
    df_ticker = df[df["Ticker"] == ticker][["Date", "Close", "Volume"]].copy()
    df_ticker = df_ticker.tail(RECENT_DAYS)
    df_ticker.rename(columns={"Date": "ds", "Close": "y"}, inplace=True)

    # Moving average
    df_ticker['MA7'] = df_ticker['y'].rolling(window=7).mean().bfill()

    # Fill NaNs in all regressors
    for col in ["y", "Volume", "MA7"]:
        df_ticker[col] = df_ticker[col].ffill().bfill()

    return df_ticker

def make_future_with_regressors(df_ticker, periods):
    """
    Create a future dataframe for Prophet, extending regressors appropriately.
    """
    future = df_ticker.copy()
    last_row = future.iloc[-1]

    # Build future dates
    future_dates = pd.date_range(start=future["ds"].iloc[-1] + pd.Timedelta(days=1),
                                 periods=periods, freq="B")
    future_extra = pd.DataFrame({"ds": future_dates})

    # Extend regressors — forward fill with last known values
    for col in ["Volume", "MA7"]:
        future_extra[col] = last_row[col]

    # Combine historical + future
    full_future = pd.concat([future, future_extra], ignore_index=True)
    return full_future

def run_backtest(df_ticker, changepoint_scale=0.2):
    """
    Rolling-origin backtest to evaluate Prophet performance.
    """
    errors = []
    last_train_index = len(df_ticker) - FORECAST_HORIZON_DAYS - 1

    for cutoff in range(last_train_index, len(df_ticker) - FORECAST_HORIZON_DAYS):
        train = df_ticker.iloc[:cutoff].copy()
        test = df_ticker.iloc[cutoff:cutoff + FORECAST_HORIZON_DAYS]

        model = Prophet(
            daily_seasonality=False,
            weekly_seasonality=False,
            yearly_seasonality=True,
            changepoint_prior_scale=changepoint_scale,
            seasonality_mode="additive",
            interval_width=0.95
        )
        model.add_regressor("Volume")
        model.add_regressor("MA7")

        model.fit(train)

        future = make_future_with_regressors(train, FORECAST_HORIZON_DAYS)
        forecast = model.predict(future)

        preds = forecast['yhat'].iloc[-FORECAST_HORIZON_DAYS:].values
        mae = mean_absolute_error(test['y'], preds)
        rmse = np.sqrt(mean_squared_error(test['y'], preds))
        errors.append({"MAE": mae, "RMSE": rmse})

    mean_mae = np.mean([e["MAE"] for e in errors])
    mean_rmse = np.mean([e["RMSE"] for e in errors])
    return mean_mae, mean_rmse

def tune_changepoint_prior_scale(df, ticker, metrics_list):
    """
    Grid search for optimal changepoint_prior_scale for each ticker.
    """
    df_ticker = prepare_data(df, ticker)
    best_mae = float("inf")
    best_cps = None

    for cps in [0.05, 0.1, 0.15, 0.2, 0.3, 0.5]:
        mae, rmse = run_backtest(df_ticker, changepoint_scale=cps)
        print(f"Ticker: {ticker}, cps={cps}, MAE={mae:.4f}, RMSE={rmse:.4f}")
        metrics_list.append({"Ticker": ticker, "Changepoint_Prior_Scale": cps, "MAE": mae, "RMSE": rmse})

        if mae < best_mae:
            best_mae = mae
            best_cps = cps

    print(f"[+] Best CPS for {ticker}: {best_cps} (MAE={best_mae:.4f})\n")
    return best_cps

def final_forecast(df, changepoint_dict):
    """
    Generate and save final forecasts for selected tickers.
    """
    forecasts = []
    for ticker in SELECTED_TICKERS:
        df_ticker = prepare_data(df, ticker)
        if len(df_ticker) < MIN_HISTORY_DAYS:
            print(f"[!] Skipping {ticker}: Insufficient history.")
            continue

        cps = changepoint_dict.get(ticker, 0.2)
        model = Prophet(
            daily_seasonality=False,
            weekly_seasonality=False,
            yearly_seasonality=True,
            changepoint_prior_scale=cps,
            seasonality_mode="additive",
            interval_width=0.95
        )
        model.add_regressor("Volume")
        model.add_regressor("MA7")

        model.fit(df_ticker)
        future = make_future_with_regressors(df_ticker, FORECAST_HORIZON_DAYS)
        forecast = model.predict(future)

        forecast["Ticker"] = ticker
        forecasts.append(forecast[["ds", "Ticker", "yhat", "yhat_lower", "yhat_upper"]])

    if forecasts:
        forecast_df = pd.concat(forecasts)
        forecast_df.to_csv("sp500_forecasts_no_sentiments.csv", index=False)
        print(f"[+] Saved sp500_forecasts_no_sentiments.csv with {len(forecast_df)} rows.")

if __name__ == "__main__":
    # Load dataset (only prices now)
    try:
        df = pd.read_csv("sp500_cleaned_data.csv", parse_dates=["Date"])
        print(f"[+] Loaded sp500_cleaned_data.csv ({len(df)} rows)")
    except FileNotFoundError:
        print("[!] Missing sp500_cleaned_data.csv")
        exit(1)

    # Tune CPS for each ticker
    metrics_list = []
    best_cps_dict = {}
    for ticker in SELECTED_TICKERS:
        if ticker not in df["Ticker"].unique():
            print(f"[!] No data for {ticker}, skipping.")
            continue
        print(f"--- Tuning {ticker} ---")
        best_cps_dict[ticker] = tune_changepoint_prior_scale(df, ticker, metrics_list)

    # Save backtest metrics
    if metrics_list:
        pd.DataFrame(metrics_list).to_csv("backtest_no_sentiments_metrics.csv", index=False)
        print(f"[+] Saved backtest_no_sentiments_metrics.csv ({len(metrics_list)} rows)")

    # Generate final forecast
    final_forecast(df, best_cps_dict)


In [ ]:
# collecting sentiments from last 1 week
import os
import time
from datetime import datetime, timedelta
import pandas as pd
from textblob import TextBlob
from yahoo_fin import news
import praw
import feedparser
import requests
import boto3
import io   # <-- new for S3 buffer writing

OUTPUT_FILE = "sentiment_last_1_week_data.csv"
NEWSAPI_KEY = os.getenv("NEWSAPI_KEY")

# ===== Load S&P 500 tickers =====
tickers = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA",
    "META", "TSLA", "PLTR", "NFLX", "AMD", "AVGO", "LLY", "GME", "UAL", "CCL", "DASH"
]

# ===== Boto3 append_to_s3 (replacement) =====
def append_to_s3(df, s3_key, bucket_name="your-bucket-name", region="us-east-1"):
    """
    Appends a DataFrame to a CSV file in S3.
    If the file doesn't exist, it creates a new one.
    """
    s3 = boto3.client("s3", region_name=region)

    try:
        # Check if object exists
        obj = s3.get_object(Bucket=bucket_name, Key=s3_key)
        existing_data = obj["Body"].read().decode("utf-8")
    except s3.exceptions.NoSuchKey:
        existing_data = ""

    # Write new rows
    csv_buffer = io.StringIO()
    df.to_csv(csv_buffer, index=False, header=not bool(existing_data))
    new_data = existing_data + csv_buffer.getvalue()

    # Upload back to S3
    s3.put_object(Bucket=bucket_name, Key=s3_key, Body=new_data.encode("utf-8"))

# ===== Reddit API =====
def get_reddit_posts(symbol):
    try:
        reddit = praw.Reddit(
            client_id="Lym7imwjxC3vDEnfildUrA",
            client_secret="QD9fuT-tuJNpMAAz4ti_uYl5zVRibA",
            user_agent="sentiment-script"
        )
        posts = []
        for subreddit in ["stocks", "investing"]:
            for post in reddit.subreddit(subreddit).search(symbol, limit=10):
                posts.append(post.title)
        return posts
    except Exception as e:
        print(f"[!] Reddit error for {symbol}: {e}")
        return []

# ===== Yahoo Finance =====
def get_yahoo_news(symbol):
    try:
        return [item["title"] for item in news.get_yf_rss(symbol)]
    except Exception as e:
        print(f"[!] Yahoo error for {symbol}: {e}")
        return []

# ===== Google News =====
def get_google_news(query):
    try:
        url = f"https://news.google.com/rss/search?q={query}&hl=en-US&gl=US&ceid=US:en"
        feed = feedparser.parse(url)
        return [entry.title for entry in feed.entries]
    except Exception as e:
        print(f"[!] Google error for {query}: {e}")
        return []

# ===== NewsAPI =====
def get_newsapi_news(query):
    if not NEWSAPI_KEY:
        return []
    try:
        url = f"https://newsapi.org/v2/everything?q={query}&language=en&pageSize=5&apiKey={NEWSAPI_KEY}"
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            data = r.json()
            return [a["title"] for a in data.get("articles", [])]
    except Exception as e:
        print(f"[!] NewsAPI error for {query}: {e}")
    return []

# ===== Sentiment Analysis =====
def analyze_sentiment(text):
    polarity = TextBlob(text).sentiment.polarity
    if polarity > 0:
        return "positive"
    elif polarity < 0:
        return "negative"
    return "neutral"

# ===== Main =====
def main():
    # Load existing data if available
    if os.path.exists(OUTPUT_FILE):
        df_existing = pd.read_csv(OUTPUT_FILE)
        df_existing["Date"] = pd.to_datetime(df_existing["Date"],errors="coerce", format="mixed")  # ensure datetime
        df_existing = df_existing.dropna(subset=["Date"])
        last_date = df_existing["Date"].max().date()
        print(f"[+] Found existing file. Last recorded date: {last_date}")
        '''df_existing = pd.read_csv(OUTPUT_FILE, parse_dates=["Date"])
        last_date = df_existing["Date"].max().date()
        print(f"[+] Found existing file. Last recorded date: {last_date}")'''
    else:
        df_existing = pd.DataFrame()
        last_date = None
        print("[!] No existing file found. Starting fresh.")

    today = datetime.utcnow().date()

    # If file exists, start from next day after last_date
    start_date = last_date + timedelta(days=1) if last_date else today - timedelta(days=7)

    if start_date > today:
        print("[+] Already up to date. Nothing to collect.")
        return

    final_rows = []

    # Loop from start_date to today
    for i in range((today - start_date).days + 1):
        current_date = (start_date + timedelta(days=i)).isoformat()
        print(f"--- Collecting sentiment for {current_date} ---")

        for ticker in tickers:
            positive, negative, neutral = 0, 0, 0
            all_texts = []

            # Collect news & posts
            sources = [
                get_reddit_posts(ticker),
                get_yahoo_news(ticker),
                get_google_news(ticker),
                get_newsapi_news(ticker)
            ]
            for src in sources:
                all_texts.extend(src)

            # Analyze sentiment
            for text in all_texts:
                s = analyze_sentiment(text)
                if s == "positive": positive += 1
                elif s == "negative": negative += 1
                else: neutral += 1

            total = positive + negative + neutral
            pos_percent = round(positive / total, 4) if total > 0 else 0

            final_rows.append([
                current_date, ticker, positive, negative, neutral, total, pos_percent
            ])

            print(f"[+] {current_date} - {ticker}: {total} mentions | Pos%: {pos_percent}")
            time.sleep(1)  # throttle to avoid bans

    # New DataFrame
    df_new = pd.DataFrame(final_rows, columns=[
        "Date","Ticker","Positive_Count","Negative_Count","Neutral_Count",
        "Total_Mentions","Positive_Percent"
    ])

    # Append to existing
    if not df_existing.empty:
        df_all = pd.concat([df_existing, df_new], ignore_index=True).drop_duplicates()
    else:
        df_all = df_new

    # Save locally
    df_all.to_csv(OUTPUT_FILE, index=False)
    print(f"[+] Updated {OUTPUT_FILE} with {len(df_all)} total rows.")

    # Append to S3 (replace bucket_name with yours!)
    append_to_s3(df_new, "sentiment_last_1_week_data.csv", bucket_name="my-sp500-stock-forecasts")
    print("[+] Updated new rows to the dataset on S3.")

if __name__ == "__main__":
    main()


In [ ]:
#forecasting using sentiments
import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings("ignore")

# ===== CONFIG =====
FORECAST_HORIZON_DAYS = 10
MIN_HISTORY_DAYS = 50
SELECTED_TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA",
    "META", "TSLA", "PLTR", "NFLX", "AMD", "AVGO", "LLY", "GME", "UAL", "CCL", "DASH"]
RECENT_DAYS = 180


def prepare_data(df, ticker):
    """
    Prepare historical data for Prophet with extra regressors.
    """
    df_ticker = df[df["Ticker"] == ticker][["Date", "Close", "Volume", "Positive_Percent"]].copy()
    df_ticker = df_ticker.tail(RECENT_DAYS)
    df_ticker.rename(columns={"Date": "ds", "Close": "y"}, inplace=True)

    # Moving average
    df_ticker["MA7"] = df_ticker["y"].rolling(window=7).mean().bfill()

    # Handle missing Positive_Percent (forward/backward fill, fallback to mean or 0)
    mean_val = df_ticker["Positive_Percent"].mean()
    df_ticker["Positive_Percent"].fillna(method="ffill", inplace=True)
    df_ticker["Positive_Percent"].fillna(method="bfill", inplace=True)
    df_ticker["Positive_Percent"].fillna(mean_val if not pd.isna(mean_val) else 0, inplace=True)

    # Fill NaNs in all regressors
    for col in ["y", "Volume", "MA7", "Positive_Percent"]:
        df_ticker[col] = df_ticker[col].ffill().bfill()

    return df_ticker


def make_future_with_regressors(df_ticker, periods):
    """
    Create a future dataframe for Prophet, extending regressors appropriately.
    """
    future = df_ticker.copy()
    last_row = future.iloc[-1]

    # Build future dates
    future_dates = pd.date_range(
        start=future["ds"].iloc[-1] + pd.Timedelta(days=1),
        periods=periods,
        freq="B"
    )
    future_extra = pd.DataFrame({"ds": future_dates})

    # Extend regressors — forward fill with last known values
    for col in ["Volume", "MA7", "Positive_Percent"]:
        future_extra[col] = last_row[col]

    # Combine historical + future
    full_future = pd.concat([future, future_extra], ignore_index=True)
    return full_future


def run_backtest(df_ticker, changepoint_scale=0.2):
    """
    Rolling-origin backtest to evaluate Prophet performance.
    """
    errors = []
    last_train_index = len(df_ticker) - FORECAST_HORIZON_DAYS - 1

    for cutoff in range(last_train_index, len(df_ticker) - FORECAST_HORIZON_DAYS):
        train = df_ticker.iloc[:cutoff].copy()
        test = df_ticker.iloc[cutoff:cutoff + FORECAST_HORIZON_DAYS]

        model = Prophet(
            daily_seasonality=False,
            weekly_seasonality=False,
            yearly_seasonality=True,
            changepoint_prior_scale=changepoint_scale,
            seasonality_mode="additive",
            interval_width=0.95
        )
        model.add_regressor("Volume")
        model.add_regressor("MA7")
        model.add_regressor("Positive_Percent")

        model.fit(train)

        future = make_future_with_regressors(train, FORECAST_HORIZON_DAYS)
        forecast = model.predict(future)

        preds = forecast["yhat"].iloc[-FORECAST_HORIZON_DAYS:].values
        mae = mean_absolute_error(test["y"], preds)
        rmse = np.sqrt(mean_squared_error(test["y"], preds))
        errors.append({"MAE": mae, "RMSE": rmse})

    mean_mae = np.mean([e["MAE"] for e in errors])
    mean_rmse = np.mean([e["RMSE"] for e in errors])
    return mean_mae, mean_rmse


def tune_changepoint_prior_scale(df, ticker, metrics_list):
    """
    Grid search for optimal changepoint_prior_scale for each ticker.
    """
    df_ticker = prepare_data(df, ticker)
    best_mae = float("inf")
    best_cps = None

    for cps in [0.05, 0.1, 0.15, 0.2, 0.3, 0.5]:
        mae, rmse = run_backtest(df_ticker, changepoint_scale=cps)
        print(f"Ticker: {ticker}, cps={cps}, MAE={mae:.4f}, RMSE={rmse:.4f}")
        metrics_list.append({"Ticker": ticker, "Changepoint_Prior_Scale": cps, "MAE": mae, "RMSE": rmse})

        if mae < best_mae:
            best_mae = mae
            best_cps = cps

    print(f"[+] Best CPS for {ticker}: {best_cps} (MAE={best_mae:.4f})\n")
    return best_cps


def final_forecast(df, changepoint_dict):
    """
    Generate and save final forecasts for selected tickers.
    """
    forecasts = []
    for ticker in SELECTED_TICKERS:
        df_ticker = prepare_data(df, ticker)
        if len(df_ticker) < MIN_HISTORY_DAYS:
            print(f"[!] Skipping {ticker}: Insufficient history.")
            continue

        cps = changepoint_dict.get(ticker, 0.2)
        model = Prophet(
            daily_seasonality=False,
            weekly_seasonality=False,
            yearly_seasonality=True,
            changepoint_prior_scale=cps,
            seasonality_mode="additive",
            interval_width=0.95
        )
        model.add_regressor("Volume")
        model.add_regressor("MA7")
        model.add_regressor("Positive_Percent")

        model.fit(df_ticker)
        future = make_future_with_regressors(df_ticker, FORECAST_HORIZON_DAYS)
        forecast = model.predict(future)

        forecast["Ticker"] = ticker
        forecasts.append(forecast[["ds", "Ticker", "yhat", "yhat_lower", "yhat_upper"]])

    if forecasts:
        forecast_df = pd.concat(forecasts)
        forecast_df.to_csv("sp500_forecasts_1weeksentiments.csv", index=False)
        print(f"[+] Saved sp500_forecasts_1weeksentiments.csv with {len(forecast_df)} rows.")


if __name__ == "__main__":
    # Load datasets
    try:
        df_prices = pd.read_csv("sp500_cleaned_data.csv", parse_dates=["Date"])
        df_prices["Date"] = pd.to_datetime(df_prices["Date"]).dt.normalize()
        print(f"[+] Loaded sp500_cleaned_data.csv ({len(df_prices)} rows)")
    except FileNotFoundError:
        print("[!] Missing sp500_cleaned_data.csv")
        exit(1)

    try:
        df_sent = pd.read_csv("sentiment_last_1_week_data.csv")
        df_sent["Date"] = pd.to_datetime(
            df_sent["Date"], errors="coerce", infer_datetime_format=True
        ).dt.normalize()
        print(f"[+] Loaded sentiment_last_1_week_data.csv ({len(df_sent)} rows)")
    except FileNotFoundError:
        print("[!] Missing sentiment_last_1_week_data.csv")
        exit(1)

    # Merge datasets
    df = pd.merge(
        df_prices,
        df_sent[["Date", "Ticker", "Positive_Percent"]],
        on=["Date", "Ticker"],
        how="left"
    )

    # Fill missing sentiment values per ticker (forward/backward fill, fallback to mean/0)
    df["Positive_Percent"] = df.groupby("Ticker")["Positive_Percent"].transform(
        lambda x: x.ffill().bfill().fillna(x.mean() if not pd.isna(x.mean()) else 0)
    )

    # Tune CPS for each ticker
    metrics_list = []
    best_cps_dict = {}
    for ticker in SELECTED_TICKERS:
        if ticker not in df["Ticker"].unique():
            print(f"[!] No data for {ticker}, skipping.")
            continue
        print(f"--- Tuning {ticker} ---")
        best_cps_dict[ticker] = tune_changepoint_prior_scale(df, ticker, metrics_list)

    # Save backtest metrics
    if metrics_list:
        pd.DataFrame(metrics_list).to_csv("backtest_metrics.csv", index=False)
        print(f"[+] Saved backtest_metrics.csv ({len(metrics_list)} rows)")

    # Generate final forecast
    final_forecast(df, best_cps_dict)


In [ ]:
import pandas as pd

df_no_sent = pd.read_csv("backtest_no_sentiments_metrics.csv")
df_sent = pd.read_csv("backtest_metrics.csv")

# Take best CPS per ticker for each method
df_no_sent_best = df_no_sent.loc[df_no_sent.groupby("Ticker")["MAE"].idxmin()]
df_sent_best = df_sent.loc[df_sent.groupby("Ticker")["MAE"].idxmin()]

# Merge to compare
comparison = pd.merge(
    df_no_sent_best[["Ticker", "MAE", "RMSE"]],
    df_sent_best[["Ticker", "MAE", "RMSE"]],
    on="Ticker",
    suffixes=("_NoSent", "_Sent")
)

# Add improvement columns
comparison["MAE_Improvement_%"] = ((comparison["MAE_NoSent"] - comparison["MAE_Sent"]) / comparison["MAE_NoSent"]) * 100
comparison["RMSE_Improvement_%"] = ((comparison["RMSE_NoSent"] - comparison["RMSE_Sent"]) / comparison["RMSE_NoSent"]) * 100

print(comparison)
comparison.to_csv("comparison_sentiment_vs_nosent.csv", index=False)


In [ ]:
from scipy.stats import ttest_rel

mae_ttest = ttest_rel(comparison["MAE_NoSent"], comparison["MAE_Sent"])
rmse_ttest = ttest_rel(comparison["RMSE_NoSent"], comparison["RMSE_Sent"])

print("MAE p-value:", mae_ttest.pvalue)
print("RMSE p-value:", rmse_ttest.pvalue)


In [ ]:
import matplotlib.pyplot as plt

# MAE comparison bar plot
comparison.set_index("Ticker")[["MAE_NoSent", "MAE_Sent"]].plot(kind="bar", figsize=(12,6))
plt.title("MAE Comparison: With vs Without Sentiment")
plt.ylabel("Mean Absolute Error")
plt.show()

# Improvement %
comparison.set_index("Ticker")[["MAE_Improvement_%", "RMSE_Improvement_%"]].plot(kind="bar", figsize=(12,6))
plt.title("Improvement % Using Sentiments")
plt.ylabel("Improvement (%)")
plt.axhline(0, color="red", linestyle="--")
plt.show()
